In [ ]:
import os
from tqdm import tqdm
from functions import load_video_to_gpu, write_to_file, Chambole_pock_TV, remove_honey_comb, frame_wise_hist_eq,

In [ ]:
#this slice goes through all files in a directory and applies the full denoising pipeline, specified in the paper

video_directory = r'C:Path\To\Your\Video\Directory'

for name in os.listdir(video_directory):
     if name[-4:] != '.avi': continue
     
     video = load_video_to_gpu(video_directory + name)
     video = Chambole_pock_TV(u_0 = video, lambd = 0.04 )        
     video = remove_honey_comb(video,kernel_size = 17)
     write_to_file(video = video, filename = video_directory+name[:-4]+'_cleaned.avi',fps = 20 )

In [ ]:
#here are some example snipets, highlighting the simplicity and adaptability of the code, depending on what part you need, i.e. only tv, filtering,histogram equalization...

In [ ]:
kernel_size = 17
lambd = 0.04

In [ ]:
#this only applies TV

target_dir = r'C:Path\To\Where\You\Want\To\Store\\'

for name in os.listdir(target_dir):
     if name[-4:] != '.avi': continue
     video = load_video_to_gpu(target_dir + name)
     video = frame_wise_hist_eq(Chambole_pock_TV(u_0 = video, lambd = lambd , use_tqdm = False))
     write_to_file(video = video, filename = target_dir+name[:-8]+'_tv.avi',fps = 20)

In [ ]:
#only filter

for name in os.listdir(target_dir):
     if name[-4:] != '.avi': continue
     video = load_video_to_gpu(target_dir + name)
     video = remove_honey_comb(video,kernel_size)
     write_to_file(video = video, filename = target_dir+name[:-8]+'_only_filt.avi',fps = 20)

In [ ]:
#only apply histogram equalization

for name in os.listdir(target_dir):
     if name[-4:] != '.avi': continue
     video = frame_wise_hist_eq(load_video_to_gpu(target_dir + name))     
     write_to_file(video = video, filename = target_dir+name[:-8]+'_raw.avi',fps = 20 )

In [ ]:
# when your GPU is to small to hold the whole video you might need to slice the video. 

from functions import  load_video_to_cpu,get_particion_idxs
import cupy as cp

file_adress = r'C:You\While\File\Adress.avi'

# In the following example a video is first loaded to cpu and then sequentially each third (num_slices = 3) of the video is placed on the GPU. 
# Depending on your hardware you will have to play around with num_slices, in order to utilize as much of the GPU as possible. (Don't forget to restart the kernel to free up the GPU)

video = load_video_to_cpu(file_adress) 
num_slices =  3       
nt,_,nx = video.shape        

for x_start,x_end in get_particion_idxs(nx,num_slices):                        
    video[:,:,x_start:x_end] = Chambole_pock_TV(u_0 = cp.array(video[:,:,x_start:x_end]), lambd = lambd , use_tqdm = False).get()   

for t_start,t_end in get_particion_idxs(nt,num_slices):                        
    video[t_start:t_end,:,:] =  remove_honey_comb(cp.array(video[t_start:t_end,:,:]),kernel_size).get()
        
write_to_file(video,file_adress[:-4]+'cleaned.avi')     
